# Session1_Task1_short — Data Loading & Exploration

In [1]:
import pandas as pd; import warnings; warnings.filterwarnings('ignore')

sales     = pd.read_csv('sales_transactions.csv')
customers = pd.read_csv('customers.csv')
products  = pd.read_csv('products.csv')

for name, df in [('sales',sales),('customers',customers),('products',products)]:
    print(f'\n=== {name} ==='); display(df.head())


=== sales ===


,transaction_id,customer_id,date,product_id,quantity,price,payment_method,channel,store_id,promotion_id,discount_amount
0,8613,683,2023-11-30,6,1,2.92,Credit Card,Online,NaN,NaN,0.00
1,1144,166,2023-11-30,15,1,3.67,Credit Card,Online,NaN,2.0,0.37
2,9130,263,2023-11-30,21,1,6.37,Mobile Pay,In-store,5.0,NaN,0.00
3,6649,576,2023-11-30,15,1,3.16,Mobile Pay,In-store,2.0,NaN,0.00
4,88,276,2023-11-30,15,1,4.54,Credit Card,Online,NaN,8.0,0.45



=== customers ===


,customer_id,first_name,last_name,age,gender,postal_code,email,phone_number,membership_status,join_date,last_purchase_date,total_spending,average_order_value,frequency,preferred_category,churned
0,101,Léa,King,60.0,M,69003,Lking@orange.fr,33765451688,Basic,14/5/2095,8/1/1960,205.92,1124.64,26,Macaron,False
1,102,Chloé,Smith,31.0,F,69006,CSMITH@HOTMAIL.COM,NaN,Gold,15/1/2074,17/9/1902,744.21,4403.42,24,Bread,False
2,103,Harper,Robinson,38.0,M,69006,harperr@yahoo.com,33148254771,Basic,1/25/2022,NaN,929.52,42.25,22,Pastries,False
3,104,Ethan,Rodriguez,54.0,F,69001,ETHAN_RODRIGUEZ@YANDEX.COM,33118046627,Basic,2033-03-23 00:00:00,7/17/2023,731.21,33.24,22,Unknown,False
4,105,Ava,Allen,50.0,F,69006,Ava.allen@hotmail.com,33267246001,Silver,16/3/2000,5/18/2023,629.48,16.57,38,Bread,True



=== products ===


,product_id,product_name,category,ingredients,price,cost,seasonal,active,introduced_date
0,1.0,Tarte Tropézienne,Tarte,"Brioche dough, pastry cream, orange blossom wa...",$6.50,$3,False,True,2018-03-25 00:00:00
1,2.0,tarte bourdaloue,Tarte,"Sweet pastry crust, almond cream, poached pear...",$7.00,$2,True,Yes,2015-10-01 00:00:00
2,3.0,tarte flambée,NaN,"Thin dough, crème fraîche, onions, bacon lardons",$8.50,$2,True,True,NaN
3,4.0,Tarte Normande,Tarte,"Sweet pastry crust, apples, Calvados custard, ...",$6.50,NaN,False,True,2033-12-23 00:00:00
4,5.0,Tarte au Citron Vert et Basilic,Tarte,"Tart shell, lime curd, fresh basil, meringue k...",$7.50,$2,No,True,NaN


In [2]:
# helper functions
def bad_dates(df, cols):   # นับวันที่ผิดปกติ
    return sum(int((pd.to_datetime(df[c],errors='coerce').pipe(lambda s: s.isna()|~s.dt.year.between(2010,2025))).sum()) for c in cols)

def bad_neg(df, cols):     # นับค่าติดลบ (รองรับ $)
    return sum(int((pd.to_numeric(df[c].astype(str).str.replace('$','',regex=False),errors='coerce')<0).sum()) for c in cols if c in df.columns)

checks = {
    'sales_transactions.csv': [bad_dates(sales,['date']), bad_neg(sales,['quantity','price','discount_amount']),
        int((~sales.customer_id.isin(customers.customer_id)|~sales.product_id.isin(products.product_id)).sum()), 0, 0],
    'customers.csv': [bad_dates(customers,['join_date','last_purchase_date']), bad_neg(customers,['total_spending','average_order_value']),
        0, int((customers.gender.notna()&~customers.gender.isin(['M','F'])).sum()),
        int(customers.email.str.contains('[A-Z]',na=False).sum())],
    'products.csv': [bad_dates(products,['introduced_date']), bad_neg(products,['price','cost']),
        0, int((products.category.notna()&~products.category.isin(['Pastries','Bread','Tarte'])).sum()),
        int(products[['price','cost']].apply(lambda c: c.astype(str).str.contains(r'\$',na=False)).sum().sum())],
}

keys = ['Invalid Dates','Negative Values','Invalid IDs','Unexpected Values','Formatting Issues']
report = ''
for name, df in [('sales_transactions.csv',sales),('customers.csv',customers),('products.csv',products)]:
    block  = f'### File: {name}\nData Types:\n'
    block += '\n'.join(f'  - {c}: {t}' for c,t in df.dtypes.items())
    block += '\n\nInconsistencies:\n'
    block += '\n'.join(f'  - {k}: {v}' for k,v in zip(keys, checks[name]))
    block += '\n' + '-'*40 + '\n'
    report += block

open('Session1_DataExploration_short.txt','w',encoding='utf-8').write(report.strip())
print('✅ Saved Session1_DataExploration_short.txt')
# จุดสังเกต: เปิดไฟล์ .txt ตรวจตัวเลขทุก section ว่าไม่เป็น 0 หมด

✅ Saved Session1_DataExploration_short.txt
